# Turkish SLAM-ASR: Görmez Hybrid Projector + Tail Augmentation + PEFT LoRA on Mozilla Common Voice (t5-ng-ta-gormez-tail-aug-lora)
### OpenAI Whisper-Large-v3 + Qwen2.5-7B-Instruct (PEFT LoRA) — İki Yönlü Çapraz Korpus Transfer Deneyi

> **Referans Makale:** *An Embarrassingly Simple Approach for LLM with Strong ASR Capacity* ([arXiv:2402.08846](https://arxiv.org/abs/2402.08846))  
> **Eğitim Veri Seti:** Mozilla Common Voice Türkçe (MDC ID: `cmu5wkah500c2o10719lem8m8`) via [Mozilla Data Collective](https://mozilladatacollective.com)  
> **Çapraz Korpus (Out-of-Domain) Test Seti:** ISSAI Turkish Speech Corpus (ISSAI_TSC - Spontane TV/Radyo Yayınları)

Bu notebook, tez/makale çalışmanızda serinin en gelişmiş mimarisi olan **Görmez Çok Kollu Projektör + Hedefli Konuşma Sonu Artırımı (Tail Augmentation) + PEFT LoRA** kombinasyonunu **Mozilla Common Voice (CV-TR)** üzerinde eğitmek ve değerlendirmek için hazırlanmıştır.

---
### Akademik Hipotez & Deney Rasyoneli:
1. **ISSAI_TSC Üzerinde Kanıtlanan Başarı:**
   - `t5-ng-ta-gormez-tail-aug-lora` mimarisi, ISSAI test kümesinde **%11.43 WER** ve sıfır atışla (zero-shot) Common Voice üzerinde **%12.75 WER** elde ederek tüm zamanların mutlak rekorunu kırmıştır.
2. **Common Voice Üzerinde Çift Yönlü Doğrulama:**
   - Şimdi aynı üstün mimari, mikrofona tane tane okunan **Common Voice** eğitim kümesi üzerinde koşturulmaktadır.
   - `cv/t0` taban çizgisi (%16.02 WER) ile karşılaştırılarak, sondan eklemeli Türkçe dil yapısında konuşma sonundaki morfolojik eklerin (sufikslerin) düşmesini engelleyen **Tail Speedup (1.15x-1.35x), Tail Energy Decay (-3 dB ile -8 dB) ve Tail SpecMasking** tekniklerinin okuma konuşmasında sağlayacağı sıçrama test edilecektir.
3. **ISSAI Spontane Yayınlara Çapraz Korpus Genellemesi:**
   - `cv/t0` taban çizgisi ISSAI üzerinde %29.60 WER vermişti. Tail Augmentation ile eğitilen modelin bu zorlu transfer açığını ne kadar kapatacağı test edilecektir.

---
### Mimari ve Bileşenler:
* **Akustik Kodlayıcı:** `openai/whisper-large-v3` (1280 dim, [DONDURULMUŞ / FROZEN]). Ham konuşma sinyalinden 50 Hz log-mel özellik çıkarır.
* **Hedefli Konuşma Sonu Veri Artırımı (Targeted Speech Tail Augmentations):**
  - Konuşmanın son %20-35'lik bölümünde hızlandırma (1.15x - 1.35x tail speedup).
  - Cümle sonundaki ses kısıklıklarını modelleyen enerji zayıflatması (-3 dB ile -8 dB decay ramp).
  - Spektrogram son dilimlerinde yoğunlaştırılmış maskeleme (Tail SpecMasking).
* **Görmez Hibrit Projektör (Projector):**
  - **Stage 1 (Akustik N-Gram Konvolüsyonu):** 1, 3, 5, 9 n-gram ardışık pencereler ile morfo-fonetik özellik çıkarımı.
  - **Stage 2 (Segmental Zamansal Dikkat Havuzlama, k=5):** 50 Hz kareleri 10 Hz'e sıkıştırır.
  - **Stage 3 (Görmez Çok Kol):** Çift yönlü GRU akustik hafıza (`GormezBiGRU`), Dual-Scale Dikkat (`Split-Heads Attention`) ve Rezidüel Kısayol (`Residual Shortcut`) paralel çalışır.
  - **Stage 4 (Füzyon ve MLP Projeksiyonu):** 1536 dim -> MLP -> 3584 dim LLM gömme uzayı.
* **Dil Modeli ve Türkçe Adaptasyon (LLM Backbone + PEFT LoRA):**
  - `Qwen/Qwen2.5-7B-Instruct` (4-bit NF4 kuantizasyon).
  - `q_proj`, `v_proj`, `k_proj`, `o_proj` katmanlarına bağlı rank=16, alpha=32 PEFT LoRA adaptörleri.


## 1. GPU Kontrolü, Google Drive ve API Anahtarları Yetkilendirmesi
Colab oturumunuzun GPU hızlandırıcıya bağlı olduğunu doğrulayın. Eğitimin ve veri indirmelerinin kesintisiz ilerlemesi, model ağırlıklarının otomatik yedeklenmesi için **Google Drive**, **HF_TOKEN** ve **MDC_API_KEY** izinlerini en baştan verin (Menü: `Runtime` -> `Change runtime type` -> `T4 GPU` veya `A100 GPU`).


In [ ]:
# GPU kullanılabilirliğini kontrol edin
!nvidia-smi

import torch
print(f"[*] PyTorch Versiyonu: {torch.__version__}")
print(f"[*] CUDA Kullanılabilir mi?: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"[*] Aktif GPU: {torch.cuda.get_device_name(0)}")
    print(f"[*] VRAM: {round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2)} GB")

# Google Drive'ı bağlayın (Eğitim bittiğinde yedeklemenin kesintisiz yapılabilmesi için izni baştan verin)
try:
    from google.colab import drive
    drive.mount("/content/drive")
    print("[*] Google Drive başarıyla bağlandı.")
except Exception as e:
    print(f"[!] Google Drive bağlantı notu: {e}")

# 3. Colab Secrets (🔑) Yetkilendirmeleri (HF_TOKEN & MDC_API_KEY)
# Model indirmelerinde hız limitine takılmamak ve veri indirmelerinin kesintisiz sürmesi için izinleri baştan verin.
import os
try:
    from google.colab import userdata

    # Hugging Face Token (Hızlı indirme ve model erişimi için)
    try:
        hf_token = userdata.get("HF_TOKEN")
        if hf_token:
            os.environ["HF_TOKEN"] = hf_token
            os.environ["HUGGING_FACE_HUB_TOKEN"] = hf_token
            print("[*] HF_TOKEN başarıyla alındı ve ortam değişkenlerine eklendi.")
    except Exception:
        print("[i] HF_TOKEN Colab Secrets'ta tanımlı değil veya izin verilmedi (İsteğe bağlı).")

    # Mozilla Data Collective API Key (Common Voice veri seti indirmesi için)
    try:
        mdc_api_key = userdata.get("MDC_API_KEY")
        if mdc_api_key:
            os.environ["MDC_API_KEY"] = mdc_api_key
            print("[*] MDC_API_KEY başarıyla alındı ve ortam değişkenlerine eklendi.")
    except Exception:
        print("[i] MDC_API_KEY Colab Secrets'ta tanımlı değil veya izin verilmedi (İsteğe bağlı).")
except ImportError:
    pass


Thu Oct 22 19:34:51 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   28C    P0             47W /  600W |       0MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## 2. Depo Dosyalarını Hazırlama ve Bağımlılıkları Yükleme
GitHub reposunu çekin veya çalışma dizinine geçin ve gerekli paketleri yükleyin.

In [ ]:
import os

# Repoyu klonlayın ve çalışma dizinine geçin
if not os.path.exists("slam_asr"):
    if not os.path.exists("slam-asr-reimplement"):
        !git clone https://github.com/gokhanersoy/slam-asr-reimplement.git
    %cd slam-asr-reimplement/tr/cv/t5-ng-ta-gormez-tail-aug-lora

# Bağımlılıkları yükleyin
!pip install -q -r requirements.txt
!pip install -q huggingface_hub soundfile torchaudio bitsandbytes accelerate datacollective mutagen peft


Cloning into 'slam-asr-reimplement'...
remote: Enumerating objects: 1742, done.
remote: Counting objects: 100% (593/593), done.
remote: Compressing objects: 100% (424/424), done.
remote: Total 1742 (delta 419), reused 332 (delta 168), pack-reused 1149 (from 1)
Receiving objects: 100% (1742/1742), 15.71 MiB | 38.96 MiB/s, done.
Resolving deltas: 100% (822/822), done.
/content/slam-asr-reimplement/tr/cv/t5-ng-ta-gormez-tail-aug-lora
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 86.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 222.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 210.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.3/71.3 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 195.7/195.7 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 9.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that a

## 3. Kod Dosyalarının Doğrulanması ve Birim Testler
Türkçe karakter normalizasyonu (`İ -> i`, `I -> ı`), projektör katmanı, dinamik batchleme ve donanım profilleme testlerini koşturun.

In [ ]:
!python run_tests.py

test_variable_length_waveform_collation (test_collator.TestCollator.test_variable_length_waveform_collation) ... ok
test_whisper_mel_collation (test_collator.TestCollator.test_whisper_mel_collation) ... ok
test_a100_40gb_tier_recommendations (test_device_utils.TestDeviceUtils.test_a100_40gb_tier_recommendations) ... ok
test_cpu_tier_recommendations (test_device_utils.TestDeviceUtils.test_cpu_tier_recommendations) ... ok
test_get_device_specs (test_device_utils.TestDeviceUtils.test_get_device_specs) ... ok
test_h100_tier_recommendations (test_device_utils.TestDeviceUtils.test_h100_tier_recommendations) ... ok
test_l4_tier_recommendations (test_device_utils.TestDeviceUtils.test_l4_tier_recommendations) ... ok
test_mps_tier_recommendations (test_device_utils.TestDeviceUtils.test_mps_tier_recommendations) ... ok
test_resolve_auto_hyperparameters (test_device_utils.TestDeviceUtils.test_resolve_auto_hyperparameters) ... ok
test_rtx_a4000_tier_recommendations (test_device_utils.TestDeviceUtil

## 4. Hızlı Doğrulama: Sentetik Türkçe Veri ile Test Eğitimi (30 Saniye)
21 GB'lık büyük veri setini indirmeden önce, tüm eğitim döngüsünün, gradyan akışının ve model ağırlıklarının hatasız çalıştığını sentetik verimizle test edin.

In [ ]:
# 1. Sentetik Türkçe test verisini üretin
!python scripts/create_toy_turkish_dataset.py

# 2. Hızlı doğrulama eğitimini koşturun (50 adım)
!python train.py --config config/train_toy_debug_turkish.yaml

[*] Generating synthetic Turkish toy dataset...
[*] Created 8 train samples in: /content/slam-asr-reimplement/tr/cv/t5-ng-ta-gormez-tail-aug-lora/data/sample_train_tr.jsonl
[*] Created 2 val samples in:   /content/slam-asr-reimplement/tr/cv/t5-ng-ta-gormez-tail-aug-lora/data/sample_val_tr.jsonl
[*] Turkish toy dataset generated successfully.
[*] HARDWARE-ADAPTIVE HYPERPARAMETER PROFILER
    Device Detected:      NVIDIA RTX PRO 6000 Blackwell Server Edition (94.97 GB VRAM), Compute sm_120
    Hardware Profile:     Ultra-High VRAM (H100 / A100 80GB)
    Profile Strategy:     Full 16-bit precision (Paper Setup). Maximum throughput without quantization.
----------------------------------------------------------------------
    Active Settings:
      • torch_dtype:       float32
      • load_in_4bit:      False (NF4)
      • batch_size:        2
      • grad_accumulation: 1  --> Effective Batch: 2
      • mixed_precision:   bf16=False, fp16=False
      • num_workers:       0
[*] Initializin

## 5. Mozilla Common Voice (Türkçe) Veri Seti İndirme ve Hazırlığı
Colab Secrets (Sol paneldeki anahtar simgesi 🔑) üzerinden `MDC_API_KEY` anahtarınız okunur ve Mozilla Data Collective API'si aracılığıyla dataset (`cmu5wkah500c2o10719lem8m8`) otomatik olarak indirilip hazırlanır.


In [ ]:
from google.colab import userdata
import os

# Colab Secrets'tan MDC_API_KEY anahtarını alın
mdc_api_key = os.environ.get("MDC_API_KEY")
if not mdc_api_key:
    try:
        mdc_api_key = userdata.get("MDC_API_KEY")
        print("[*] MDC_API_KEY anahtari Colab Secrets uzerinden basariyla alindi.")
    except Exception:
        mdc_api_key = input("Lutfen Mozilla Data Collective API anahtarinizi (MDC_API_KEY) girin: ")

# Mozilla Common Voice Türkçe veri setini indirin ve hazırlayın
!python scripts/prepare_common_voice.py \
    --download \
    --api_key "{mdc_api_key}" \
    --output_dir data/cv


[*] Connecting to Mozilla Data Collective REST API...
[*] Requesting presigned download URL from Mozilla Data Collective API...
    URL: https://mozilladatacollective.com/api/datasets/cmu5wkah500c2o10719lem8m8/download
[*] Downloading dataset to: /content/slam-asr-reimplement/tr/cv/t5-ng-ta-gormez-tail-aug-lora/data/cv/common-voice-scripted-speech-27-0-turkis-b14760fe.tar.gz
    Progress: 0% (8.0 MB / 2853.8 MB)
    Progress: 5% (144.0 MB / 2853.8 MB)
    Progress: 10% (288.0 MB / 2853.8 MB)
    Progress: 15% (432.0 MB / 2853.8 MB)
    Progress: 20% (576.0 MB / 2853.8 MB)
    Progress: 25% (720.0 MB / 2853.8 MB)
    Progress: 30% (864.0 MB / 2853.8 MB)
    Progress: 35% (1000.0 MB / 2853.8 MB)
    Progress: 40% (1144.0 MB / 2853.8 MB)
    Progress: 45% (1288.0 MB / 2853.8 MB)
    Progress: 50% (1432.0 MB / 2853.8 MB)
    Progress: 55% (1576.0 MB / 2853.8 MB)
    Progress: 60% (1720.0 MB / 2853.8 MB)
    Progress: 65% (1856.0 MB / 2853.8 MB)
    Progress: 70% (2000.0 MB / 2853.8 MB)
   

## 6. Donanıma Göre Otomatik Optimize Olan Türkçe SLAM-ASR Eğitimi (Görmez + Tail Aug + LoRA)
Donanım profilleyicimiz (`device_utils`) GPU'nuzun VRAM kapasitesini tespit ederek batch size, gradient accumulation ve veri tiplerini optimize eder.
Eğitim sırasında hedefli konuşma sonu artırımı (Tail Speedup, Energy Decay, Tail SpecMasking) aktif olarak uygulanır.


In [ ]:
# Donanıma göre otomatik optimize edilen Türkçe SLAM-ASR eğitimi
!python train.py --config config/hardware_adaptive_turkish.yaml

[*] HARDWARE-ADAPTIVE HYPERPARAMETER PROFILER
    Device Detected:      NVIDIA RTX PRO 6000 Blackwell Server Edition (94.97 GB VRAM), Compute sm_120
    Hardware Profile:     Ultra-High VRAM (H100 / A100 80GB)
    Profile Strategy:     Full 16-bit precision (Paper Setup). Maximum throughput without quantization.
----------------------------------------------------------------------
    Active Settings:
      • torch_dtype:       bfloat16
      • load_in_4bit:      False (NF4)
      • batch_size:        16
      • grad_accumulation: 1  --> Effective Batch: 16
      • mixed_precision:   bf16=True, fp16=False
      • num_workers:       4
[*] Initializing SLAM-ASR with config: config/hardware_adaptive_turkish.yaml
config.json: 100% 1.27k/1.27k [00:00<00:00, 7.78MB/s]

model.safetensors: downloading bytes:   6% 186M/3.09G [00:01<00:09, 303MB/s, 13.1MB/s  ]
model.safetensors: reconstructing file:   4% 134M/3.09G [00:01<00:24, 121MB/s]
model.safetensors: downloading bytes:   8% 258M/3.09G [00

## 7. Model Değerlendirmesi: Mozilla Common Voice Test Seti (In-Domain)
Eğitilen Görmez + Tail Aug + LoRA modelinin Common Voice test seti üzerindeki performansını ölçün. Türkçe büyük/küçük harf (`İ -> i`, `I -> ı`) ve noktalama normalizasyonu otomatik uygulanır.


In [ ]:
!python evaluate.py \
    --config config/hardware_adaptive_turkish.yaml \
    --checkpoint outputs/slam_asr_cv_turkish_adaptive/best_checkpoint \
    --manifest data/cv/test.jsonl \
    --output_file eval_results.jsonl \
    --language tr \
    --beam_size 4


[*] Loading model on cuda...
Loading weights: 100% 1259/1259 [00:00<00:00, 15615.26it/s]
Loading weights: 100% 339/339 [00:00<00:00, 15107.78it/s]
[*] Successfully attached LoRA adapters to LLM backbone.
[*] Loading projector weights from: outputs/slam_asr_cv_turkish_adaptive/best_checkpoint/projector.pt
[*] Loading trained LoRA adapter from: outputs/slam_asr_cv_turkish_adaptive/best_checkpoint/lora_adapter
[*] Successfully loaded LoRA adapter.
[*] Evaluating 11874 samples with beam_size=4...
Evaluating:   0% 0/743 [00:00<?, ?it/s]/usr/local/lib/python3.13/dist-packages/transformers/generation/utils.py:1464: UserWarning: Passing `repetition_penalty` with `inputs_embeds` and without `input_ids` to `generate` will apply the penalty only to newly generated tokens, not to the prompt.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/utils.py:1472: UserWarning: Passing `no_repeat_ngram_size` with `inputs_embeds` and without `input_ids` to `generate` will apply

## 8. İnteraktif Çıkarım (Ses Dinleme ve Transkripsiyon)
Test setindeki bir Common Voice ses kaydını dinleyin ve modelinizin konuşmayı nasıl yazıya döktüğünü canlı olarak görün.


In [ ]:
import json
import os
import IPython.display as ipd

# Test setinden ilk örneği bulun
sample_audio = None
sample_text = ""

manifest_file = "data/cv/test.jsonl" if os.path.exists("data/cv/test.jsonl") else "data/sample_val_tr.jsonl"
if os.path.exists(manifest_file):
    with open(manifest_file, "r", encoding="utf-8") as f:
        first_line = f.readline()
        if first_line:
            data = json.loads(first_line)
            sample_audio = data.get("audio_filepath")
            sample_text = data.get("text", "")

if sample_audio and os.path.exists(sample_audio):
    print(f"[*] Seçilen Ses: {sample_audio}")
    print(f"[*] Gerçek Metin (Ground Truth): {sample_text}\n")
    display(ipd.Audio(sample_audio))

    !python inference.py \
        --audio "{sample_audio}" \
        --config config/hardware_adaptive_turkish.yaml \
        --checkpoint outputs/slam_asr_cv_turkish_adaptive/best_checkpoint \
        --prompt "Konuşmayı metne dönüştürün."
else:
    print("[!] Test ses dosyası bulunamadı. Lütfen ses dosyasının yolunu elle belirtin.")


[*] Seçilen Ses: /content/slam-asr-reimplement/tr/cv/t5-ng-ta-gormez-tail-aug-lora/data/cv/extracted/cv-corpus-27.0-2026-09-11/tr/clips/common_voice_tr_21461823.mp3
[*] Gerçek Metin (Ground Truth): belgrad da aynı derecede kuşkulu görünüyor



[*] Loading SLAM-ASR on cuda...
Loading weights: 100% 1259/1259 [00:00<00:00, 15546.76it/s]
Loading weights: 100% 339/339 [00:00<00:00, 14475.63it/s]
[*] Successfully attached LoRA adapters to LLM backbone.
[*] Loading projector weights from: outputs/slam_asr_cv_turkish_adaptive/best_checkpoint/projector.pt
[*] Loading trained LoRA adapter from: outputs/slam_asr_cv_turkish_adaptive/best_checkpoint/lora_adapter
[*] Successfully loaded LoRA adapter.
[*] Processing audio: /content/slam-asr-reimplement/tr/cv/t5-ng-ta-gormez-tail-aug-lora/data/cv/extracted/cv-corpus-27.0-2026-09-11/tr/clips/common_voice_tr_21461823.mp3
[*] Transcribing (beam_size=4)...
/usr/local/lib/python3.13/dist-packages/transformers/generation/utils.py:1464: UserWarning: Passing `repetition_penalty` with `inputs_embeds` and without `input_ids` to `generate` will apply the penalty only to newly generated tokens, not to the prompt.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/utils.py:

## 9. Zaman Damgalı Google Drive Yedeklemesi
Eğitilen ağırlıkları (`best_checkpoint` içindeki projektör ve LoRA adaptörü), test sonuçlarını (`eval_results.jsonl`), manifestoları ve konfigürasyonu Google Drive'a tarih/saat damgalı olarak kalıcı yedekleyin.


In [ ]:
# 9. Adım: Tüm Model Ağırlıklarını, LoRA Adaptörlerini ve Test Sonuçlarını Drive'a Yedekleme
import os
from datetime import datetime
from google.colab import drive

# 1. Google Drive bağlantısını kontrol edin (Henüz bağlı değilse bağlayın)
if not os.path.exists("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")

# 2. Zaman damgalı hedef klasörü belirleyin
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
drive_target_dir = f"/content/drive/MyDrive/checkpoints/{timestamp}_slam_asr_tr_cv_t5_ng_ta_gormez_tail_aug_lora"
os.makedirs(drive_target_dir, exist_ok=True)

# 3. Model Ağırlıklarını (Projektör + LoRA) Kopyalayın
!cp -r outputs/slam_asr_cv_turkish_adaptive/best_checkpoint "{drive_target_dir}/"
if os.path.exists("outputs/slam_asr_cv_turkish_adaptive/final_checkpoint"):
    !cp -r outputs/slam_asr_cv_turkish_adaptive/final_checkpoint "{drive_target_dir}/"

# 4. (Varsa) Değerlendirme Çıktısını Kopyalayın
if os.path.exists("eval_results.jsonl"):
    !cp eval_results.jsonl "{drive_target_dir}/"

# 5. Veri Manifestolarını Kopyalayın (train/val/test jsonl)
if os.path.exists("data/cv"):
    os.makedirs(f"{drive_target_dir}/manifests", exist_ok=True)
    !cp data/cv/*.jsonl "{drive_target_dir}/manifests/"

# 6. Kullanılan Config Dosyasını Kopyalayın
!cp config/hardware_adaptive_turkish.yaml "{drive_target_dir}/"

print("\n" + "=" * 65)
print(f"[*] Tam Yedekleme Başarıyla Tamamlandı!")
print(f"[*] Hedef Dizin: {drive_target_dir}")
print("    • best_checkpoint & final_checkpoint (Görmez Projektör + LoRA)")
print("    • eval_results.jsonl (Test tahminleri ve WER analizi)")
print("    • train/val/test manifestoları")
print("    • hardware_adaptive_turkish.yaml konfigürasyonu")
print("=" * 65)



[*] Tam Yedekleme Başarıyla Tamamlandı!
[*] Hedef Dizin: /content/drive/MyDrive/checkpoints/20261023_024350_slam_asr_tr_cv_t5_ng_ta_gormez_tail_aug_lora
    • best_checkpoint & final_checkpoint (Görmez Projektör + LoRA)
    • eval_results.jsonl (Test tahminleri ve WER analizi)
    • train/val/test manifestoları
    • hardware_adaptive_turkish.yaml konfigürasyonu


### 10. Çapraz Korpus Testi: ISSAI Turkish Speech Corpus (ISSAI_TSC) ile Değerlendirme (Out-of-Domain Generalization)
Common Voice Türkçe verisiyle eğitilen Görmez + Tail Aug + LoRA modelinin, eğitimde hiç görmediği **ISSAI Turkish Speech Corpus (ISSAI_TSC)** test kümesi (spontane konuşmalar, TV/radyo yayınları, duraksamalar, ses çakışmaları) üzerindeki alan dışı (out-of-domain) genelleme başarımı test edilir.

> **Akademik Değerlendirme:** `cv/t0` taban çizgisi bu testte **%29.60 WER** vermişti. Hedefli konuşma sonu artırımı (Tail Augmentation) ve Görmez Projektörün bu zorlu alandaki transfer performansı ölçülecektir.


In [ ]:
# 10. Adım: Common Voice ile Eğitilen Modelin ISSAI_TSC Test Seti Üzerinde Değerlendirilmesi
import os
import glob
import json
import re

# 1. Drive'ı Bağlayın (Henüz bağlı değilse)
if not os.path.exists("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")

# 2. Model checkpoint'ini belirleyin (Önce yerel, sonra Drive)
checkpoint_path = "outputs/slam_asr_cv_turkish_adaptive/best_checkpoint"
if not os.path.exists(checkpoint_path):
    candidates = sorted(glob.glob("/content/drive/MyDrive/checkpoints/*slam_asr_tr_cv*/best_checkpoint") + glob.glob("/content/drive/MyDrive/checkpoints/*slam_asr_cv_turkish*/best_checkpoint"), key=lambda p: (re.search(r"(\d{8}_\d{6})", p).group(1) if re.search(r"(\d{8}_\d{6})", p) else "", p))
    if candidates:
        checkpoint_path = candidates[-1]

# 3. ISSAI_TSC test manifestosunu bulun ve ses dosyalarının diskte var olduğunu doğrulayın
def is_valid_manifest(path):
    if not path or not os.path.exists(path):
        return False
    try:
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                if not line.strip():
                    continue
                data = json.loads(line)
                audio_path = data.get("audio_filepath", "")
                return os.path.exists(audio_path)
    except Exception:
        return False
    return False

tsc_manifest = None
manifest_candidates = [
    "data/issai_tsc/test.jsonl",
    "../../tsc/t0/data/issai_tsc/test.jsonl",
    "../tsc/data/issai_tsc/test.jsonl",
    "/content/slam-asr-reimplement/tr/tsc/t0/data/issai_tsc/test.jsonl",
    "/content/slam-asr-reimplement/tr/tsc/data/issai_tsc/test.jsonl",
]
manifest_candidates.extend(sorted(glob.glob("/content/drive/MyDrive/checkpoints/*slam_asr_tr_tsc*/manifests/test.jsonl")))

for p in manifest_candidates:
    if is_valid_manifest(p):
        tsc_manifest = p
        break

# 4. ISSAI test ses dosyaları diskte yoksa otomatik indir/hazırla
if not tsc_manifest:
    prep_script = "scripts/prepare_issai_tsc.py"
    if not os.path.exists(prep_script):
        for cand in ["../../tsc/t0/scripts/prepare_issai_tsc.py", "../tsc/scripts/prepare_issai_tsc.py", "/content/slam-asr-reimplement/tr/tsc/t0/scripts/prepare_issai_tsc.py", "/content/slam-asr-reimplement/tr/tsc/scripts/prepare_issai_tsc.py"]:
            if os.path.exists(cand):
                prep_script = cand
                break
    if os.path.exists(prep_script):
        try:
            from google.colab import userdata
            hf_token = userdata.get("HF_TOKEN")
        except Exception:
            hf_token = None
        hf_flag = f'--hf_token "{hf_token}"' if hf_token else ""
        print("[*] ISSAI_TSC test verisi yerel diskte bulunamadı. Hugging Face Hub üzerinden hazırlanıyor...")
        !python "{prep_script}" --download {hf_flag} --output_dir data/issai_tsc
        if is_valid_manifest("data/issai_tsc/test.jsonl"):
            tsc_manifest = "data/issai_tsc/test.jsonl"

# 5. Değerlendirmeyi çalıştır ve sonucu Drive'a kaydet
if tsc_manifest and os.path.exists(checkpoint_path):
    print(f"[*] Kullanılan Checkpoint: {checkpoint_path}")
    print(f"[*] ISSAI_TSC Test Manifestosu: {tsc_manifest}\n")
    !python evaluate.py \
        --config config/hardware_adaptive_turkish.yaml \
        --checkpoint "{checkpoint_path}" \
        --manifest "{tsc_manifest}" \
        --output_file eval_results_tsc_generalization.jsonl \
        --language tr \
        --beam_size 4

    # Değerlendirme çıktısını Drive'daki checkpoint klasörüne yedekle
    drive_target = None
    if "/content/drive/MyDrive" in checkpoint_path:
        drive_target = os.path.dirname(checkpoint_path.rstrip("/"))
    elif "drive_target_dir" in globals() and os.path.exists(drive_target_dir):
        drive_target = drive_target_dir

    if drive_target and os.path.exists(drive_target) and os.path.exists("eval_results_tsc_generalization.jsonl"):
        !cp eval_results_tsc_generalization.jsonl "{drive_target}/"
        print(f"\n[*] ISSAI_TSC değerlendirme sonuçları Drive'a yedeklendi: {drive_target}/eval_results_tsc_generalization.jsonl")
else:
    if not os.path.exists(checkpoint_path):
        print(f"[!] Model checkpoint'i ({checkpoint_path}) bulunamadı. Lütfen önce 6. adımı çalıştırıp modeli eğitin veya Drive'daki bir checkpoint'i kontrol edin.")
    if not tsc_manifest:
        print("[!] ISSAI_TSC test ses dosyaları yerel diskte bulunamadı.")
        print("    (Drive'da test.jsonl manifestosu bulunsa dahi, içindeki ses dosyaları mevcut Colab diskinde yok.)")
        print("    İpucu: Colab Secrets'a (🔑) 'HF_TOKEN' ekleyerek ISSAI verisini otomatik indirebilir veya")
        print("    'data/issai_tsc/test.jsonl' yolunu elle belirtebilirsiniz.")


[*] ISSAI_TSC test verisi yerel diskte bulunamadı. Hugging Face Hub üzerinden hazırlanıyor...
[*] Downloading ISSAI_TSC_218.tar.gz from huggingface.co/datasets/issai/Turkish_Speech_Corpus...

ISSAI_TSC_218.tar.gz: downloading bytes:   1% 194M/21.4G [00:01<01:14, 284MB/s, 10.0MB/s  ]
ISSAI_TSC_218.tar.gz: reconstructing file:   0% 67.1M/21.4G [00:01<06:56, 51.2MB/s]
ISSAI_TSC_218.tar.gz: downloading bytes:   2% 497M/21.4G [00:01<00:39, 528MB/s, 37.4MB/s  ]
ISSAI_TSC_218.tar.gz: downloading bytes:   3% 614M/21.4G [00:01<00:37, 547MB/s, 52.2MB/s  ]
ISSAI_TSC_218.tar.gz: downloading bytes:   4% 892M/21.4G [00:02<00:31, 647MB/s, 73.6MB/s  ]
ISSAI_TSC_218.tar.gz: downloading bytes:   5% 973M/21.4G [00:02<00:29, 685MB/s, 80.4MB/s  ]
ISSAI_TSC_218.tar.gz: downloading bytes:   7% 1.39G/21.4G [00:02<00:24, 826MB/s,  114MB/s  ]
ISSAI_TSC_218.tar.gz: downloading bytes:  13% 2.74G/21.4G [00:04<00:15, 1.19GB/s,  217MB/s  ]
ISSAI_TSC_218.tar.gz: downloading bytes:  14% 3.01G/21.4G [00:04<00:14, 1.29G

## 11. Modeli Belleğe Yükle ve Canlı Mikrofon Kaydı ile Konuşmayı Metne Dönüştür
Model GPU belleğine bir kere yüklenir ve hazır bekler. Ardından tarayıcınızdan mikrofona Türkçe konuşursunuz ve model konuşmanızı 1 saniye içinde yazıya döker.


In [ ]:
# 11. Adım: Modeli Belleğe Yükle ve Mikrofon ile Canlı Konuşmayı Metne Dönüştür
import os
import glob
import subprocess
import yaml
from base64 import b64decode
import torch
import IPython.display as ipd
from google.colab import output

from slam_asr.models.slam_model import SLAM_ASR, SLAMASRConfig
from slam_asr.data.audio_processor import AudioProcessor
from slam_asr.utils.device_utils import resolve_auto_hyperparameters

# 1. Drive'ı Bağla ve En Güncel Checkpoint'i Bul
if not os.path.exists("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")

checkpoint_candidates = sorted(glob.glob("/content/drive/MyDrive/checkpoints/*slam_asr_tr_cv*/best_checkpoint/projector.pt"), key=lambda p: (re.search(r"(\d{8}_\d{6})", p).group(1) if re.search(r"(\d{8}_\d{6})", p) else "", p))

if checkpoint_candidates:
    best_ckpt = checkpoint_candidates[-1]
elif os.path.exists("outputs/slam_asr_cv_turkish_adaptive/best_checkpoint/projector.pt"):
    best_ckpt = "outputs/slam_asr_cv_turkish_adaptive/best_checkpoint/projector.pt"
else:
    raise FileNotFoundError("Hiçbir best_checkpoint/projector.pt dosyası bulunamadı!")

# 2. Modeli GPU'ya Yükle (Zaten Yüklüyse Tekrar Yüklemez!)
if "loaded_slam_model" not in globals():
    print(f"[*] Model checkpoint'i kullanılıyor:\n    {best_ckpt}\n")
    print("[*] 1/3: Model GPU VRAM'ine yükleniyor... Lütfen bekleyin...")

    cfg = yaml.safe_load(open("config/hardware_adaptive_turkish.yaml"))
    model_cfg = cfg.get("model", {})
    data_cfg = cfg.get("data", {})
    model_cfg, data_cfg, _ = resolve_auto_hyperparameters(model_cfg, data_cfg, {})

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model_config = SLAMASRConfig(**model_cfg)

    model = SLAM_ASR(config=model_config)
    if not getattr(model_config, "load_in_4bit", False) and not getattr(model_config, "load_in_8bit", False):
        model = model.to(device)
    else:
        model.speech_encoder = model.speech_encoder.to(device)
        model.projector = model.projector.to(device)

    state_dict = torch.load(best_ckpt, map_location=device)
    model.projector.load_state_dict(state_dict)

    lora_dir = os.path.join(os.path.dirname(best_ckpt), "lora_adapter")
    if os.path.exists(lora_dir) and os.path.exists(os.path.join(lora_dir, "adapter_config.json")):
        try:
            from peft import PeftModel
            print(f"[*] LoRA adaptörü yükleniyor: {lora_dir}")
            if hasattr(model.language_model, "peft_config"):
                model.language_model.load_adapter(lora_dir, adapter_name="default")
            else:
                model.language_model = PeftModel.from_pretrained(model.language_model, lora_dir)
            print("[*] LoRA adaptörü başarıyla yüklendi.")
        except Exception as e:
            print(f"[!] Uyarı: LoRA adaptörü yüklenemedi ({e})")

    model.eval()

    audio_proc = AudioProcessor(
        sampling_rate=data_cfg.get("sampling_rate", 16000),
        is_whisper=data_cfg.get("is_whisper", True),
        whisper_mel_bins=data_cfg.get("whisper_mel_bins", 128),
    )

    loaded_slam_model = model
    loaded_audio_proc = audio_proc
    loaded_device = device
    loaded_model_config = model_config
    print("[*] 2/3: Model başarıyla VRAM'e yüklendi ve hazır bekliyor! 🚀\n")
else:
    print("[*] Model zaten VRAM'de hazır bekliyor (Yeniden yükleme yapılmadı). 🚀\n")

# 3. Model Hazır Olduktan Sonra Mikrofon Butonunu Göster
RECORD_JS = """
const sleep = time => new Promise(resolve => setTimeout(resolve, time));
const b2text = blob => new Promise(resolve => {
  const reader = new FileReader();
  reader.onloadend = e => resolve(e.target.result);
  reader.readAsDataURL(blob);
});
var recordAudio = () => new Promise(async (resolve, reject) => {
  try {
    const stream = await navigator.mediaDevices.getUserMedia({ audio: true });
    const recorder = new MediaRecorder(stream);
    const chunks = [];
    recorder.ondataavailable = e => chunks.push(e.data);

    const container = document.createElement("div");
    container.style.cssText = "padding: 12px; margin: 10px 0; background: #1e1e2e; border-radius: 8px; color: white; font-family: sans-serif; display: inline-block;";

    const btn = document.createElement("button");
    btn.innerHTML = "🔴 Kaydı Başlat (Konuşmaya Başlayın)";
    btn.style.cssText = "padding: 12px 24px; font-size: 16px; font-weight: bold; background: #e74c3c; color: white; border: none; border-radius: 6px; cursor: pointer; transition: 0.3s; box-shadow: 0 4px 6px rgba(0,0,0,0.3);";
    container.appendChild(btn);
    document.body.appendChild(container);

    let isRecording = false;

    btn.onclick = () => {
      if (!isRecording) {
        recorder.start();
        isRecording = true;
        btn.innerHTML = "⏹️ Kaydı Bitir (Sonucu Gör)";
        btn.style.background = "#27ae60";
      } else {
        recorder.stop();
        btn.innerHTML = "⏳ Model transkribe ediyor...";
        btn.disabled = true;
        btn.style.background = "#7f8c8d";
      }
    };

    recorder.onstop = async () => {
      const blob = new Blob(chunks, { type: "audio/webm" });
      const base64Text = await b2text(blob);
      container.remove();
      resolve(base64Text);
    };
  } catch (err) {
    reject(err);
  }
});
recordAudio();
"""

print("[*] 3/3: Mikrofon hazır. Konuşmak için aşağıdaki butona basın:")
audio_data = output.eval_js(RECORD_JS)

# 4. Sesi 16 kHz Mono WAV Formatına Dönüştür
webm_file = "live_temp.webm"
wav_file = "live_mic_recording.wav"

with open(webm_file, "wb") as f:
    f.write(b64decode(audio_data.split(",")[1]))

subprocess.run([
    "ffmpeg", "-y", "-i", webm_file,
    "-ar", "16000", "-ac", "1",
    wav_file
], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

print("\n[*] Kaydettiğiniz Ses:")
display(ipd.Audio(wav_file))

# 5. Bellekteki Model ile Anında Çıkarım
speech_tensor = loaded_audio_proc.process(wav_file).unsqueeze(0).to(loaded_device)
model_dtype_str = getattr(loaded_model_config, "dtype", None) or getattr(loaded_model_config, "torch_dtype", "float32")
use_amp = (model_dtype_str in ("float16", "bfloat16")) and loaded_device.type == "cuda"
amp_dtype = torch.bfloat16 if model_dtype_str == "bfloat16" else torch.float16

with torch.no_grad():
    if use_amp:
        autocast_ctx = (
            torch.amp.autocast("cuda", dtype=amp_dtype)
            if hasattr(torch, "amp") and hasattr(torch.amp, "autocast")
            else torch.cuda.amp.autocast(dtype=amp_dtype)
        )
        with autocast_ctx:
            transcription = loaded_slam_model.generate(
                speech_inputs=speech_tensor,
                prompts=["Konuşmayı metne dönüştürün."],
                num_beams=4,
            )[0]
    else:
        transcription = loaded_slam_model.generate(
            speech_inputs=speech_tensor,
            prompts=["Konuşmayı metne dönüştürün."],
            num_beams=4,
        )[0]

print("\n" + "=" * 55)
print(f"🗣️  Sizin Konuşmanız: {transcription}")
print("=" * 55)


[*] Model checkpoint'i kullanılıyor:
    /content/drive/MyDrive/checkpoints/20261023_024350_slam_asr_tr_cv_t5_ng_ta_gormez_tail_aug_lora/best_checkpoint/projector.pt

[*] 1/3: Model GPU VRAM'ine yükleniyor... Lütfen bekleyin...
[*] HARDWARE-ADAPTIVE HYPERPARAMETER PROFILER
    Device Detected:      NVIDIA RTX PRO 6000 Blackwell Server Edition (94.97 GB VRAM), Compute sm_120
    Hardware Profile:     Ultra-High VRAM (H100 / A100 80GB)
    Profile Strategy:     Full 16-bit precision (Paper Setup). Maximum throughput without quantization.
----------------------------------------------------------------------
    Active Settings:
      • torch_dtype:       bfloat16
      • load_in_4bit:      False (NF4)
      • batch_size:        16
      • grad_accumulation: 1  --> Effective Batch: 16
      • mixed_precision:   bf16=True, fp16=False
      • num_workers:       4


Loading weights:   0%|          | 0/1259 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

[*] Successfully attached LoRA adapters to LLM backbone.
[*] LoRA adaptörü yükleniyor: /content/drive/MyDrive/checkpoints/20261023_024350_slam_asr_tr_cv_t5_ng_ta_gormez_tail_aug_lora/best_checkpoint/lora_adapter
[*] LoRA adaptörü başarıyla yüklendi.
[*] 2/3: Model başarıyla VRAM'e yüklendi ve hazır bekliyor! 🚀

[*] 3/3: Mikrofon hazır. Konuşmak için aşağıdaki butona basın:
